In [2]:
from google.colab import files
uploaded = files.upload()

Saving Dataset .csv to Dataset .csv


In [4]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Dataset ni load cheyadam
df = pd.read_csv('Dataset .csv')
df['Cuisines'] = df['Cuisines'].fillna('Unknown')
df.head(2)

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591


In [5]:
df['content'] = df['Cuisines'] + ' ' + df['City'] + ' PriceRange_' + df['Price range'].astype(str)
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['content'])
print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (9551, 317)


In [7]:
def recommend_restaurants(preferred_cuisine, preferred_city, preferred_price_range=None, top_n=5):
    query = f"{preferred_cuisine} {preferred_city}"
    if preferred_price_range is not None:
        query += f" PriceRange_{preferred_price_range}"
    query_vec = tfidf.transform([query])
    similarity_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = similarity_scores.argsort()[::-1][:50]
    matched = df.iloc[top_indices].copy()
    matched['Similarity'] = similarity_scores[top_indices]
    ranked = matched.sort_values(by=['Similarity', 'Aggregate rating', 'Votes'], ascending=[False, False, False])
    display_cols = ['Restaurant Name', 'City', 'Cuisines', 'Price range', 'Aggregate rating', 'Votes']
    return ranked[display_cols].head(top_n)

In [8]:
print("=== Sample 1: North Indian in New Delhi (Price range 2) ===")
results_1 = recommend_restaurants(preferred_cuisine='North Indian', preferred_city='New Delhi', preferred_price_range=2, top_n=5)
display(results_1)
print("\n=== Sample 2: Cafe / Desserts in Pune ===")
results_2 = recommend_restaurants(preferred_cuisine='Cafe Desserts', preferred_city='Pune', top_n=5)
display(results_2)

=== Sample 1: North Indian in New Delhi (Price range 2) ===


,Restaurant Name,City,Cuisines,Price range,Aggregate rating,Votes
3015,Anand Restaurant,New Delhi,North Indian,2,3.6,485
4835,Oberoi's,New Delhi,North Indian,2,3.6,75
6213,Kake Da Hotel,New Delhi,North Indian,2,3.6,29
3041,Kake Da Hotel,New Delhi,North Indian,2,3.5,2185
7451,Galina Restaurant,New Delhi,North Indian,2,3.5,38



=== Sample 2: Cafe / Desserts in Pune ===


,Restaurant Name,City,Cuisines,Price range,Aggregate rating,Votes
9163,Saut��ed Stories,Pune,"Cafe, Italian, Continental",2,4.2,583
9170,Tales & Spirits,Pune,"Italian, Continental, Cafe",2,4.1,997
9157,Le Plaisir,Pune,"European, Desserts",3,4.8,2510
9173,Barbeque Ville,Pune,North Indian,3,4.4,208
9167,Barbeque Nation,Pune,"North Indian, Mughlai",3,4.5,2847
